# Bangladesh Food Price Watch — run the project in Google Colab

Master's **project** (not a thesis): detect, (locate) and explain unexplained food price rises in Dhaka TCB daily retail prices.

**How to use this notebook**
1. Open this notebook from GitHub: `https://colab.research.google.com/github/mehediparvez/foodprice_watch/blob/main/FoodPriceWatch_Colab.ipynb`. Cell 2 clones the code into `MyDrive/FoodPriceWatch/` (and pulls updates on later runs).
2. Put the 32 project data files in `MyDrive/FoodPriceWatch/inbox/project/` (data is not in the public repo).
3. Run the cells top to bottom. `Runtime → Change runtime type → CPU` is enough (no GPU needed).

**If Colab disconnects or the network drops:** reconnect, run cells **1 → 4** again (cell 2 also pulls any code fixes), then re-run the stage cell that was interrupted.
Nothing finished is recomputed: every unit of work is saved on Drive with a `_SUCCESS.json` marker, and stages marked `done`
in `state/stages.json` are skipped. The test periods are sealed: a resumed run re-uses the same seal instead of counting a second look.

| Cell | Stage | Typical time (2 vCPU) |
|---|---|---|
| 5 | S00 intake + git | < 1 min |
| 6 | S01 rebuild TCB panel | < 1 min |
| 7 | S04 driver panel (availability lags, leakage tests) | 1–2 min |
| 8 | S05 reproduce the classical baseline (87 forecast origins) | **~1.5–2.5 h**, resumable |
| 9 | S06 LightGBM benchmark + driver groups + placebo (tune on 2024 → pre-register → test once) | ~10–20 min |
| 10 | S07 intervals and alerts (pre-register on 2024 → score events once) | ~5–10 min |
| 11 | S09 explanation ladder + event studies | ~10–20 min |
| 12 | S10 robustness | ~5–10 min |
| 13 | S11 results pack | < 1 min |

S02 (audit) and S03/S08 (WFP intake and locate) stay **blocked** until the original TCB files / external datasets are supplied; their code is not part of this version.

## 1. Mount Google Drive and set the project root
If the mount fails: Runtime → Disconnect and delete runtime, re-run, and tick **all** permission boxes in the popup (or use a personal Gmail account). If Drive still will not mount, this cell falls back to local storage — then back up with the cell at the end before leaving.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
DRIVE_OK = False
if IN_COLAB:
    from google.colab import drive
    for attempt in range(2):
        try:
            drive.mount("/content/drive", force_remount=True, timeout_ms=300000); DRIVE_OK = True; break
        except Exception as e:
            print(f"Drive mount attempt {attempt+1} failed: {e}")
if DRIVE_OK or not IN_COLAB:
    ROOT = os.environ.get("FPW_ROOT", "/content/drive/MyDrive/FoodPriceWatch")
    else:
    print("Using LOCAL storage /content/FoodPriceWatch (survives reconnects, NOT runtime deletion/timeout). Back up with the last cell.")
    ROOT = "/content/FoodPriceWatch"
os.makedirs(ROOT, exist_ok=True)
print("ROOT =", ROOT, "| Drive mounted:", DRIVE_OK)

## 2. Get the code from GitHub (first run: clone; later runs: pull updates)
Code comes from `github.com/mehediparvez/foodprice_watch`. Your data (`inbox/`, `data/`), results (`outputs/`, `reports/`) and run state (`state/`) live in the same Drive folder; pulling only updates code. Your stage commits stay local.

In [ ]:
import subprocess, pathlib
REPO = "https://github.com/mehediparvez/foodprice_watch.git"
BRANCH = "main"
root = pathlib.Path(ROOT)
def git(*args, check=True):
    r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if check and r.returncode: raise RuntimeError("git " + " ".join(args) + " failed")
    return r
if not (root / ".git").exists():
    git("init", "-q"); git("remote", "add", "origin", REPO)
    git("fetch", "--depth", "50", "origin", BRANCH)
    git("checkout", "-f", "-B", BRANCH, f"origin/{BRANCH}")       # inbox/, data/, outputs/ are git-ignored, so your files are kept
    print("Cloned code into", ROOT)
else:
    git("config", "user.email", "fpw@example.com"); git("config", "user.name", "FPW runner")
    git("stash", "-u", check=False)                                    # protect anything uncommitted
    git("pull", "--rebase", "origin", BRANCH, check=False)             # code updates; your local stage commits are replayed on top
    git("stash", "pop", check=False)
git("config", "user.email", "fpw@example.com"); git("config", "user.name", "FPW runner")
git("log", "--oneline", "-3")
for d in ["inbox/project", "inbox/external", "inbox/tcb_originals", "data/raw", "data/processed", "outputs", "reports/tables", "reports/figures", "state/seals", "logs"]:
    (root / d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT); print("cwd:", os.getcwd())
missing = [f for f in ["tcb_dhaka_daily_prices_2021_2026.csv", "item_meta.csv", "imf_commodity_prices.csv"] if not (root / "inbox/project" / f).exists() and not (root / "data/raw/project" / f).exists()]
if missing: print("\n>>> Put the 32 project data files in", root / "inbox/project", "— missing e.g.", missing)

## 3. Install packages and import everything (re-run after every reconnect)

In [ ]:
import subprocess
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements-colab.txt"], check=True)
import json, glob, time, platform, importlib
import numpy as np, pandas as pd, matplotlib, yaml
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
versions = {"python": platform.python_version()}
for m in ["numpy", "pandas", "scipy", "matplotlib", "sklearn", "statsforecast", "lightgbm", "shap", "pyarrow", "yaml", "tabulate", "pytest", "openpyxl"]:
    try: versions[m] = importlib.import_module(m).__version__
    except Exception as e: versions[m] = f"MISSING ({e})"
versions["cpus"] = os.cpu_count()
print(json.dumps(versions, indent=1))
os.chdir(ROOT); sys.path.insert(0, f"{ROOT}/src")
os.environ["PYTHONPATH"] = f"{ROOT}/src"
os.environ.setdefault("FPW_THREADS", str(os.cpu_count() or 2))
pathlib.Path("state/env_versions.json").write_text(json.dumps(versions, indent=2))   # recorded for reproducibility

## 4. Helpers + startup ritual (re-run after every reconnect)
`make check` equivalent: re-hashes every finished stage's outputs, resets any stage whose files changed, runs all tests.

In [ ]:
def sh(cmd, check=True):
    """Run a shell command in ROOT and stream its output."""
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    for line in p.stdout:
        if "Warning" in line or "warnings.warn" in line: continue
        print(line, end="")
    p.wait()
    if check and p.returncode: raise RuntimeError(f"FAILED ({p.returncode}): {cmd}")
    return p.returncode
def ledger():
    return {s["id"]: s for s in json.loads(pathlib.Path(ROOT, "state/stages.json").read_text())["stages"]}
def status():
    d = pd.DataFrame(ledger().values())[["id", "title", "status", "units_done", "last_unit", "finished_at", "notes"]]
    display(d)
def run_stage(sid, cmds, note=""):
    """Skip finished stages; otherwise run each command (each is itself resumable) and finalize (tests -> done -> commit)."""
    st = ledger()[sid]["status"]
    if st == "done": print(f"{sid} already done — skipped."); return
    if st == "blocked": print(f"{sid} is blocked: {ledger()[sid]['notes']}"); return
    deps = [d for d in ledger()[sid]["depends_on"] if ledger()[d]["status"] not in ("done", "skipped")]
    if deps: print(f"WARNING: {sid} depends on {deps} which are not done.")
    for c in cmds:
        print(f"\n$ {c}"); sh(c)
    sh(f'python src/stages/finalize.py {sid} {note}')
# --- startup ritual ---
os.chdir(ROOT)
if not pathlib.Path(".git").exists():
    sh("git init -q && git config user.email 'fpw@example.com' && git config user.name 'FPW runner'")
for t in glob.glob("outputs/**/*.tmp", recursive=True) + glob.glob("data/**/*.tmp", recursive=True) + glob.glob("state/*.tmp"):
    os.remove(t); print("removed leftover", t)
if pathlib.Path("data/raw/MANIFEST.csv").exists():
    sh("python -m fpw.check", check=False)
status()

## 5. S00 — intake: copy inbox → data/raw, hash everything, mark blocked stages, first commit

In [ ]:
sh("python src/stages/s00_intake.py")
sh("git add -A && git commit -qm 'S00: scaffold and intake' || true")
run_stage("S00", [])
status()

## 6. S01 — rebuild the TCB panel (invariants, proposed Palm Olein correction, T1, F1, F2)

In [ ]:
run_stage("S01", ["python -W ignore src/stages/s01.py"], "Palm Olein 2L proposed correction not applied")
print(json.dumps(json.loads(pathlib.Path("outputs/s01_checks.json").read_text()), indent=1, default=str)[:1500])
display(pd.read_csv("reports/tables/T1.csv")); display(Image("reports/figures/F1_indexed_staples.png", width=800))

## 7. S04 — driver panel (slots G0–G5, explanation drivers E1–E5, availability lags, Comtrade +208 days, leakage tests)

In [ ]:
run_stage("S04", ["python -W ignore src/stages/s04.py"])
display(pd.read_csv("reports/tables/T3.csv"))
print(pathlib.Path("config/s04_frozen.json").read_text())

## 8. S05 — reproduce the existing classical baseline (long; resumable)
One unit = one forecast origin (7 statsforecast models × 54 series). If Colab disconnects, re-run cells 1–4 and this cell: finished origins are skipped.
If the reproduction differs from `baseline_results_by_group.csv` by more than 0.02, the test asks for `outputs/05_baseline/REPRO_EXPLAINED.md`. The reference run already found such differences (AutoTheta h=1, AutoARIMA) — see `docs/REPRO_EXPLAINED_reference.md`. If your `repro_diff_ALL.csv` shows the same pattern, copy that file to `outputs/05_baseline/REPRO_EXPLAINED.md` (after reading it); otherwise write your own explanation. Then re-run this cell.

In [ ]:
run_stage("S05", ["python -W ignore src/stages/s05_run.py test", "python -W ignore src/stages/s05_score.py"])   # tip: "s05_run.py test 10" stops after 10 new origins
display(pd.read_csv("outputs/05_baseline/repro_diff_ALL.csv").round(3))

## 9. S06 — LightGBM benchmark and driver value (RQ1, RQ2)
Order is enforced: tune on 2024 only → write pre-registration → **commit** → unseal the 2025–26 test once → score.

In [ ]:
def s06():
    st = ledger()["S06"]["status"]
    if st == "done": print("S06 already done — skipped."); return
    sh("python -W ignore src/stages/s06_tune.py")
    if not pathlib.Path("state/prereg_S06.yaml").exists():
        sh("python -W ignore src/stages/s06_prereg.py")
    sh("git add -A && git commit -qm 'S06: pre-registration committed before unsealing' || true")
    run_stage("S06", ["python -W ignore src/stages/s06_run.py", "python -W ignore src/stages/s06_score.py"])
s06()
t4 = pd.read_csv("reports/tables/T4.csv"); display(t4[t4.eval_stratum == "ALL"].pivot(index="model", columns="step", values="gmean_relMAE").round(3))
display(pd.read_csv("reports/tables/T5b.csv").round(4)); display(Image("reports/figures/F4_driver_value.png", width=800))

## 10. S07 — intervals and alerts (RQ3)
Real-time h=7 expected prices → operating point chosen on 2024 non-event false alarms → pre-registration **committed** → events scored once.

In [ ]:
def s07():
    if ledger()["S07"]["status"] == "done": print("S07 already done — skipped."); return
    sh("python -W ignore src/stages/s07_run.py")
    if not pathlib.Path("state/prereg_S07.yaml").exists():
        sh("python -W ignore src/stages/s07_prereg.py")
    sh("git add -A && git commit -qm 'S07: pre-registration committed before event scoring' || true")
    run_stage("S07", ["python -W ignore src/stages/s07_score.py"])
s07()
display(pd.read_csv("reports/tables/T7_event_level.csv")); t8 = pd.read_csv("reports/tables/T8.csv"); display(t8[t8.eval_stratum == "ALL"])
display(Image("reports/figures/F6_alert_timelines.png", width=900))

## 11. S09 — explanation ladder M0 ⊂ M1 ⊂ M2, SHAP, sign checks, event studies (RQ4)

In [ ]:
run_stage("S09", ["python -W ignore src/stages/s09_run.py", "python -W ignore src/stages/s09_score.py"])
display(pd.read_csv("reports/tables/T9.csv").round(3)); display(pd.read_csv("reports/tables/T10.csv").round(4))
display(Image("reports/figures/F8_decomposition.png", width=800))

## 12. S10 — robustness (placebo event dates, leave-one-event-out, confidence subsets, sliding window, ginger candidates)

In [ ]:
run_stage("S10", ["python -W ignore src/stages/s10.py"])
display(pd.read_csv("reports/tables/T11.csv").round(3))

## 13. S11 — results pack (RESULTS_REGISTER.md + README.md). Optional exact re-run diff: add `--rerun-diff` (~30 min).

In [ ]:
run_stage("S11", ["python -W ignore src/stages/s11.py"])
display(Markdown(pathlib.Path("reports/README.md").read_text()))
status()

## 14. Look at every figure

In [ ]:
for f in sorted(glob.glob("reports/figures/*.png")):
    print(f); display(Image(f, width=850))

## Notes
- **Never edit `data/raw/`.** Corrections go to `data/processed/corrections_proposed.csv` and need your approval.
- **Decisions:** `DECISIONS.md` holds defaults awaiting your confirmation. If you change one (e.g. the alert false-alarm target), log it in `DEVIATIONS.md`, delete the affected stage's outputs and pre-registration, set its status to `pending`, and re-run — a second unsealing of a test needs an approved deviation.
- **Author:** `config.yaml → author` is `TO_CONFIRM`; no name is written into any output until you set it.
- **Wording:** residual rises are always "not explained by measured cost drivers" (`tests/test_wording.py` enforces this).

## 15. Backup (needed only when running on local storage instead of Drive)

In [ ]:
import shutil; shutil.make_archive("/content/fpw_backup", "zip", ROOT)
if IN_COLAB:
    from google.colab import files; files.download("/content/fpw_backup.zip")
# Restore later: upload fpw_backup.zip, then  !mkdir -p /content/FoodPriceWatch && unzip -o -q fpw_backup.zip -d /content/FoodPriceWatch